# Next architecture: frozen EfficientNet-B0 (E2)

This notebook builds the next model in the project's experiment plan: **E2, transfer learning with a frozen EfficientNet-B0 backbone**. It follows E1 (`SimpleCNN`, trained from scratch) and reuses the repository's model, data, and training code rather than creating a separate implementation.

The goal is to classify a face image as real (`0`) or AI-generated (`1`). The notebook walks through the project and data, visualizes the architecture, checks available data, instantiates and verifies the model, and provides an **opt-in** small training run. Training does not start automatically.

> **Colab notes:** select a GPU runtime in **Runtime → Change runtime type** for faster training. This notebook can clone the project into the Colab runtime if needed. The dataset download is about 4 GB; Colab's local files disappear when its runtime is reset, so consider saving results to Google Drive in the optional cell below.

> No data counts or model results are fabricated; they are computed from the files you download.


## 1. What is already in the project?

- **E1 / baseline:** `SimpleCNN` has four convolution → batch-normalization → ReLU → max-pooling blocks and is trained from random initialization.
- **E2 / this notebook:** ImageNet-pretrained EfficientNet-B0 is used as a feature extractor. Its backbone parameters and BatchNorm running statistics are frozen; only its new one-logit classifier head is trained.
- The planned sequence then compares full fine-tuning (E3), gradual unfreezing (E4), ResNet-50 (E5), and degradation-aware training (E6).

The choice of E2 is grounded in the experiment plan: it is the next experiment after the scratch baseline and asks whether pretrained visual features are useful without changing the backbone. The dataset documentation describes Kaggle's `xhlulu/140k-real-and-fake-faces`: 70,000 real and 70,000 StyleGAN-generated faces, with documented train/validation/test splits of 100,000 / 20,000 / 20,000. The expected layout is `data/raw/real_vs_fake/real-vs-fake/{train,valid,test}/{real,fake}`. Actual local counts are checked below rather than assumed.

**Label and evaluation rules:** `fake` is the positive class (`1`), `real` is `0`, and the model estimates `P(fake)`. Use training data for fitting, validation data for model selection, and keep the test set untouched until final evaluation. Do not use customer or other private identity data.

## 2. Architecture at a glance

```mermaid
flowchart LR
    A[RGB face image\n224 x 224 x 3] --> B[ImageNet normalization\nmean and standard deviation]
    B --> C[EfficientNet-B0 feature extractor\npretrained weights, frozen]
    C --> D[Global pooling / feature vector]
    D --> E[Dropout 0.2]
    E --> F[New linear classifier\n1 logit]
    F --> G[Sigmoid]
    G --> H[P(fake)\nthreshold 0.5 by default]
```

**Text fallback:** normalized `224×224×3` image → frozen pretrained EfficientNet-B0 → pooled feature vector → dropout (`0.2`) → new linear layer (one logit) → sigmoid probability of fake. `timm` constructs the backbone and classifier; the exact feature width is read from the instantiated model rather than hard-coded. During E2, only the classifier head receives weight updates.

## 3. Locate the project, load E2 settings, and check the data

Run the cells in order. Everything needed to prepare the environment and data is provided in notebook cells; you do not need to use a terminal. If this notebook was opened directly in Colab rather than from the repository folder, the first cell clones the project from GitHub.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess

REPO_NAME = "UOT_SCS_3546_041_Deep-Learning"
REPO_URL = "https://github.com/msondhi/UOT_SCS_3546_041_Deep-Learning.git"
def find_project_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "src" / "deepfake_kyc").is_dir() and (
            candidate / "configs" / "experiments" / "02_effnet_frozen.yaml"
        ).is_file():
            return candidate
    return None

PROJECT_ROOT = find_project_root()
if PROJECT_ROOT is None:
    try:
        import google.colab  # noqa: F401
    except ImportError as exc:
        raise FileNotFoundError(
            "Could not locate the repository. Open the notebook from the project folder, "
            "or run it in Colab so it can clone the repository."
        ) from exc
    PROJECT_ROOT = Path("/content") / REPO_NAME
    if not PROJECT_ROOT.exists():
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_ROOT)], check=True)
    if not (PROJECT_ROOT / "src" / "deepfake_kyc").is_dir() or not (
        PROJECT_ROOT / "configs" / "experiments" / "02_effnet_frozen.yaml"
    ).is_file():
        raise FileNotFoundError(f"Repository clone is missing expected project files: {PROJECT_ROOT}")

sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")

### Notebook-only dependency setup

This cell installs only project dependencies that are missing from the active kernel. It deliberately does not install or replace PyTorch/torchvision, which are supplied by Colab. It does not start training.

In [ ]:
import importlib.util

required_modules = {
    "timm": "timm",
    "numpy": "numpy",
    "PIL": "Pillow",
    "sklearn": "scikit-learn",
    "yaml": "PyYAML",
    "tqdm": "tqdm",
    "matplotlib": "matplotlib",
}
missing_packages = [package for module, package in required_modules.items()
                    if importlib.util.find_spec(module) is None]
if missing_packages:
    print(f"Installing missing packages: {', '.join(missing_packages)}")
    subprocess.run([sys.executable, "-m", "pip", "install", *missing_packages], check=True)
else:
    print("All notebook and project dependencies are already available.")
try:
    import torch
    import torchvision
except ImportError as exc:
    raise ImportError(
        "This Colab runtime is missing PyTorch or torchvision. Select a standard "
        "Colab Python runtime, then restart and rerun the notebook."
    ) from exc
print(f"PyTorch {torch.__version__} | torchvision {torchvision.__version__}")
print(f"CUDA GPU available: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    print("For faster training, choose Runtime > Change runtime type > T4 GPU, then rerun the notebook.")

In [ ]:
from deepfake_kyc.config import load_config

CONFIG_PATH = PROJECT_ROOT / "configs" / "experiments" / "02_effnet_frozen.yaml"
cfg = load_config(CONFIG_PATH)
DATA_ROOT = PROJECT_ROOT / cfg["data"]["root"]
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
print(f"Experiment: {cfg['experiment_name']} | model: {cfg['model']['name']}")
print(f"Expected data root: {DATA_ROOT}")
split_paths = {name: DATA_ROOT / folder for name, folder in (("train", "train"), ("validation", "valid"), ("test", "test"))}
for split, path in split_paths.items():
    print(f"{split:10} {'available' if path.is_dir() else 'MISSING'}: {path}")
DATA_AVAILABLE = all(path.is_dir() for path in split_paths.values())

### Optional: keep model outputs in Google Drive

Colab's `/content` storage is temporary. Enable this cell to mount Drive and save checkpoints, metrics, and configuration files under `MyDrive/deepfake_kyc_outputs/`. If you leave it disabled, outputs remain in the current Colab runtime and should be downloaded before disconnecting.

In [ ]:
SAVE_OUTPUTS_TO_DRIVE = True
if SAVE_OUTPUTS_TO_DRIVE:
    try:
        from google.colab import drive
    except ImportError as exc:
        raise RuntimeError("Google Drive output saving is available only in Google Colab.") from exc
    drive.mount("/content/drive")
    TRAIN_OUTPUT_DIR = Path("/content/drive/MyDrive/deepfake_kyc_outputs")
else:
    TRAIN_OUTPUT_DIR = PROJECT_ROOT / "outputs"
TRAIN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Training outputs will be saved to: {TRAIN_OUTPUT_DIR}")

### Optional: download the dataset from this notebook

The Kaggle dataset is about 4 GB. To download it here, first accept the dataset's terms on Kaggle and create API credentials. Set `DOWNLOAD_DATA = True` below; the cell prompts for your Kaggle username and API key without displaying the key, installs the Kaggle client if needed, and extracts files into the repository's ignored `data/raw/` folder. Credentials are held only in this kernel's environment and are not written to the notebook. Leave the switch `False` if the dataset is already present or you do not want to download it.

In [ ]:
DOWNLOAD_DATA = False  # Change to True only when you want to download the ~4 GB dataset.
if DOWNLOAD_DATA and not DATA_AVAILABLE:
    from getpass import getpass
    import importlib.util

    os.environ["KAGGLE_USERNAME"] = input("Kaggle username: ").strip()
    os.environ["KAGGLE_KEY"] = getpass("Kaggle API key (input hidden): ")
    if not os.environ["KAGGLE_USERNAME"] or not os.environ["KAGGLE_KEY"]:
        raise ValueError("Both Kaggle username and API key are required.")
    if importlib.util.find_spec("kaggle") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "kaggle", "six"], check=True)
    RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        [sys.executable, "-m", "kaggle", "datasets", "download", "-d",
         "xhlulu/140k-real-and-fake-faces", "-p", str(RAW_DATA_DIR), "--unzip"],
        check=True,
    )
    DATA_AVAILABLE = all(path.is_dir() for path in split_paths.values())
    if not DATA_AVAILABLE:
        raise FileNotFoundError(f"Download finished but expected split folders were not found under {DATA_ROOT}.")
    print(f"Dataset ready at {DATA_ROOT}")
elif DOWNLOAD_DATA and DATA_AVAILABLE:
    print("Dataset splits are already available; no download needed.")
else:
    print("Dataset download skipped.")

## 4. Inspect the dataset, if it is available

The counts below are computed from the actual files using the same `ImageFolder`-based dataset and label remapping used by project training. A few examples are displayed only when the folders exist. If the dataset is missing, the cell reports that fact and does not invent counts.

In [ ]:
if DATA_AVAILABLE:
    from collections import Counter
    import matplotlib.pyplot as plt
    from deepfake_kyc.data import make_dataset

    datasets_by_split = {}
    for split, path in split_paths.items():
        dataset = make_dataset(path)
        datasets_by_split[split] = dataset
        fake_idx = dataset.class_to_idx["fake"]
        counts = Counter(int(target == fake_idx) for _, target in dataset.samples)
        print(f"{split:10} total={len(dataset):,} | real (0)={counts[0]:,} | fake (1)={counts[1]:,}")

    sample_dataset = datasets_by_split["train"]
    shown = min(6, len(sample_dataset))
    if shown == 0:
        raise ValueError("The training folder exists but contains no supported images.")
    figure, axes = plt.subplots(1, shown, figsize=(3 * shown, 3))
    if shown == 1:
        axes = [axes]
    for index, axis in enumerate(axes):
        image, label = sample_dataset[index]
        axis.imshow(image)
        axis.set_title("fake (1)" if label == 1 else "real (0)")
        axis.axis("off")
    plt.tight_layout()
else:
    print("Skipped: actual image files are unavailable. Expected folders are shown above.")

## 5. Reproducibility and compute device

The experiment config sets seed `42`. The helper below applies the project's seed policy and chooses an available device (CUDA, Apple MPS, or CPU). EfficientNet-B0 with pretrained weights can require an initial weight download; this notebook will not initiate training unless the opt-in cell is run.

In [ ]:
from deepfake_kyc.utils import get_device, seed_everything

seed_everything(cfg["seed"])
device = get_device()
print(f"Seed: {cfg['seed']} | device: {device}")

## 6. Build and verify the E2 model

`build_model` delegates to `timm.create_model` with `num_classes=1`, replacing the original ImageNet classifier. We freeze the backbone with the repository helper, then check trainable parameter counts and the output shape on a dummy batch. The dummy tensor verifies wiring only; it is not data or a performance result.

In [ ]:
import torch
from deepfake_kyc.models import build_model, set_backbone_trainable, split_parameters

model = build_model(
    cfg["model"]["name"],
    pretrained=cfg["model"]["pretrained"],
    dropout=cfg["model"]["dropout"],
).to(device)
set_backbone_trainable(model, trainable=False)
backbone_params, head_params = split_parameters(model)
total_parameters = sum(parameter.numel() for parameter in model.parameters())
trainable_parameters = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
print(f"Total parameters: {total_parameters:,}")
print(f"Backbone parameters (frozen): {sum(p.numel() for p in backbone_params):,}")
print(f"Classifier-head parameters (trainable): {sum(p.numel() for p in head_params):,}")
if any(parameter.requires_grad for parameter in backbone_params):
    raise RuntimeError("E2 expects every backbone parameter to be frozen.")
if not head_params or not all(parameter.requires_grad for parameter in head_params):
    raise RuntimeError("E2 expects a present, trainable classifier head.")

model.eval()
with torch.no_grad():
    dummy_batch = torch.zeros(2, 3, cfg["data"]["image_size"], cfg["data"]["image_size"], device=device)
    logits = model(dummy_batch)
print(f"Dummy batch output shape: {tuple(logits.shape)} (expected: (2, 1))")
if tuple(logits.shape) != (2, 1):
    raise RuntimeError(f"Expected one logit per image, got {tuple(logits.shape)}")
print("Each logit is converted to P(fake) with sigmoid during prediction.")

## 7. How E2 is trained

The project trainer uses binary cross-entropy with logits (`BCEWithLogitsLoss`) and AdamW. E2 uses the YAML override `train.freeze_backbone_epochs=999`, so the backbone parameters and BatchNorm running statistics stay fixed while the classifier head learns. Training images use the project's standard augmentation; validation images use deterministic resizing and ImageNet normalization. The scheduler, validation metrics, early stopping, and checkpoint writing all come from `deepfake_kyc.train` and its shared helpers.

The next cell is **opt-in** and starts a short development run only if all data splits exist. It uses a reproducible 1,000-image training subset, 300-image validation subset, one epoch, and zero data-loader workers. Adjust these after a successful smoke run. Do not train on or tune against the test split.

In [ ]:
# OPT-IN: run this cell only after reviewing the settings and confirming the dataset is available.
if not DATA_AVAILABLE:
    raise FileNotFoundError(
        f"Training data is missing under {DATA_ROOT}. Set DOWNLOAD_DATA=True in the notebook's dataset cell, run it, then rerun the data check."
    )

from deepfake_kyc.train import main as train_experiment

development_overrides = [
    "data.train_subset=1000",
    "data.val_subset=300",
    "data.num_workers=0",
    "train.epochs=1",
    "train.freeze_backbone_epochs=999",
    "experiment_name=02_effnet_frozen_notebook_demo",
    f"output_dir={TRAIN_OUTPUT_DIR}",
]
summary = train_experiment([
    "--config", str(CONFIG_PATH),
    "--set", *development_overrides,
])
summary

## 8. Read the validation artifacts

The trainer writes `best.pt`, `history.csv`, `summary.json`, and the resolved `config.yaml` under `TRAIN_OUTPUT_DIR/02_effnet_frozen_notebook_demo/`. The cell below reads the summary/history if a run has completed. These are **validation** results only; keep the test set untouched until you are finished choosing the model.

In [ ]:
import csv
import json

run_dir = TRAIN_OUTPUT_DIR / "02_effnet_frozen_notebook_demo"
summary_path = run_dir / "summary.json"
history_path = run_dir / "history.csv"
if summary_path.is_file() and history_path.is_file():
    print(json.dumps(json.loads(summary_path.read_text()), indent=2))
    with history_path.open(newline="", encoding="utf-8") as history_file:
        for row in csv.DictReader(history_file):
            print(row)
else:
    print(f"No completed notebook training run found at {run_dir}. Run the opt-in cell first.")

### Optional: final held-out test evaluation

Run this only once after you have finished model selection. It evaluates the saved checkpoint on the test split and writes `eval_test.csv` beside the checkpoint. Do not use these test results to tune the model.

In [ ]:
RUN_FINAL_TEST_EVALUATION = False  # Change to True only for the final held-out test evaluation.
if RUN_FINAL_TEST_EVALUATION:
    from deepfake_kyc.evaluate import main as evaluate_experiment

    best_checkpoint = TRAIN_OUTPUT_DIR / "02_effnet_frozen_notebook_demo" / "best.pt"
    test_results_path = best_checkpoint.parent / "eval_test.csv"
    if not best_checkpoint.is_file():
        raise FileNotFoundError(f"Trained checkpoint not found: {best_checkpoint}")
    if test_results_path.exists():
        raise FileExistsError(
            f"Test results already exist at {test_results_path}; refusing to evaluate the test set again."
        )
    final_test_results = evaluate_experiment(["--checkpoint", str(best_checkpoint)])
    final_test_results

## 9. What comes next?

After E2, E3 (`03_effnet_finetune.yaml`) fine-tunes the entire EfficientNet-B0 backbone with a lower learning rate for pretrained features. Compare E2 and E3 with the same seed and data, choose settings using validation results, then run the optional notebook test cell once on the held-out test set. The project also recommends robustness tests (blur, JPEG compression, downscaling, noise, screen capture) and an out-of-distribution test using faces from a different generator.

**Limitations and responsible use:** this research model distinguishes examples from the dataset; it does not establish a person's identity or prove fraud. Results may not transfer to unseen generators, demographic groups, cameras, or real KYC conditions. Do not use it as the sole basis for consequential decisions, and do not process private identity/customer data.